# Assignment 3 - Write a U-Net for salt identification
## Due date - December 14, 2020

For the salt identification, the list of files provided for this course are:

1. ./data/seis -> seismic images for train dataset
2. ./data/mask -> mask images for train dataset

The mask classifies each pixel of the seismic images as salt or not. So, this is a problem of object recognition, and an [Image Segmentation](https://lmb.informatik.uni-freiburg.de/people/ronneber/u-net/) may be the recommended method. For this, the Keras package will be used to create, train, and validate the model.


In [ ]:
import pandas as pd
import numpy as np
import os
import cv2      # ! pip install opencv-python (if package is not installed)
from random import *
from tensorflow.keras import Model, Input
from tensorflow.keras.models import load_model, save_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
from tensorflow.keras.layers import MaxPooling2D, Conv2D, Dense, Dropout, Conv2DTranspose
from tensorflow.keras.layers import UpSampling2D, BatchNormalization, Activation, Add
from tensorflow.keras.layers import concatenate
from sklearn.model_selection import train_test_split
import datetime
# Add for GPU BEFORE JSON
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession

config = ConfigProto()
config.gpu_options.allow_growth = True
session = InteractiveSession(config=config)
import matplotlib.pyplot as plt
%matplotlib inline

from importlib import reload
from tensorflow.keras import backend as K

Setting data folders names and list of training files.

In [ ]:
TRAIN_IMAGE_DIR = './data/seis/' #img_id is x(input)
TRAIN_MASK_DIR = './data/mask/'   #rle_mask is y(output)

train_d = os.listdir(TRAIN_IMAGE_DIR)

Loading train data, figures and masks.

In [ ]:
x = [np.array(cv2.imread(TRAIN_IMAGE_DIR + p, cv2.IMREAD_GRAYSCALE), dtype=np.uint8) for p in train_d]
x = np.array(x)/255

y = [np.array(cv2.imread(TRAIN_MASK_DIR + p, cv2.IMREAD_GRAYSCALE), dtype=np.uint8) for p in train_d]
y = np.array(y)/255
print(x.shape,y.shape)

Resizing images for a power of 2 ($2^n$) to stabilize the U-net:

In [ ]:
def resizeImage(img, size = 128):
    imgR = np.zeros((img.shape[0],size,size))
    for i in range(img.shape[0]):
        imgR[i] = cv2.resize(img[i],(size,size))
    return(imgR)

In [ ]:
x = resizeImage(x)
y = resizeImage(y)

Ploting the images of seismic sessions and salt classification:

In [ ]:
def plotImageTransp(file1, file2, k, alpha = 0.2):
    fig, ax = plt.subplots(nrows=k, ncols=k, figsize=(18, 18))
    for i in range(k):
        for j in range(k):
            ind = randint(0,file1.shape[0]-1)
            ax[i,j].imshow(file1[ind], cmap='Greys')
            ax[i,j].imshow(file2[ind], cmap='Greens', alpha = alpha)
            ax[i,j].set_axis_off()
    fig.subplots_adjust(wspace = -0.15, hspace = 0.02)
    return(fig)

In [ ]:
fig = plotImageTransp(x, y, k = 8)

Expand arrays dimension for Keras:

In [ ]:
x = np.expand_dims(x, axis = 3) #EXPAND DIM OF X AND INSERT NEW AXIS @ 3 
y = np.expand_dims(y, axis = 3)
print(x.shape, y.shape)

### Spliting data into train and validation sets

In [ ]:
x_train, x_valid, y_train, y_valid = train_test_split(x, y, test_size = 0.2, random_state = 666)
print(x_train.shape, y_train.shape, x_valid.shape, y_valid.shape)

### Data augmentation (run if you desire to, but it will require more computer power)

Create "new" images from the existent ones.

In [ ]:
# x_train = np.append(x_train, [np.fliplr(x) for x in x_train], axis=0)
# y_train = np.append(y_train, [np.fliplr(x) for x in y_train], axis=0)
# x_train = np.append(x_train, [np.flipud(x) for x in x_train], axis=0)
# y_train = np.append(y_train, [np.flipud(x) for x in y_train], axis=0)

In [ ]:
# fig, axs = plt.subplots(4, 10, figsize=(22,8))
# for i in range(10):
#     axs[0][i].imshow(x_train[i].squeeze(), cmap="Greys")
#     axs[0][i].imshow(y_train[i].squeeze(), cmap="Greens", alpha=0.3)
#     axs[1][i].imshow(x_train[int(len(x_train)/4 + i)].squeeze(), cmap="Greys")
#     axs[1][i].imshow(y_train[int(len(y_train)/4 + i)].squeeze(), cmap="Greens", alpha=0.2)
#     axs[2][i].imshow(x_train[int(len(x_train)/2 + i)].squeeze(), cmap="Greys")
#     axs[2][i].imshow(y_train[int(len(y_train)/2 + i)].squeeze(), cmap="Greens", alpha=0.2)
#     axs[3][i].imshow(x_train[int(len(x_train)/4*3 + i)].squeeze(), cmap="Greys")
#     axs[3][i].imshow(y_train[int(len(y_train)/4*3 + i)].squeeze(), cmap="Greens", alpha=0.2)
# fig.suptitle("Top row: original images, bottom 3 rows: augmented images")

In [ ]:
print(x_train.shape, y_train.shape)

### Creating the *Keras* model

In [ ]:
def unet_simple(pretrained_weights = None, input_size = (128,128,1)):

    inputs = Input(input_size)
    
    # 128
    conv1 = Conv2D(2, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(inputs)
    conv1 = BatchNormalization()(conv1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)
    
    # Center: 128 -> 64
    conv2 = Conv2D(4, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool1)
    conv2 = BatchNormalization()(conv2)
    drop2 = Dropout(0.5)(conv2)

    # 64 -> 128
    up3 = Conv2DTranspose(2,(2,2),strides=(2,2), activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(drop2)
    up3 = BatchNormalization()(up3)
    merge3 = concatenate([conv1,up3],axis=3)
    conv3 = Conv2D(2, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge3)
    conv3 = BatchNormalization()(conv3)
    conv4 = Conv2D(1, 1, activation = 'sigmoid')(conv3)

    model = Model(inputs = inputs, outputs = conv4)

    if(pretrained_weights):
        model.load_weights(pretrained_weights)

    return model

## Exercise

The `unet_simple()` it too simple. For a better training and predictions, you need to create a more complex model. So, create a model with the following instructions:

1. 4 convolutional layers for the *encoder* part, 1 layer for the *compressed representation*, and 4 more layers for the *decoder*
2. Each convolutional layer **MUST** contain at least 1 `Conv2D`, 1 `BatchNormalization`. Remember to include a `Conv2DTranspose` and a `concatenate` to the decoder layers.
3. Set the `padding` parameter of the `Conv2D` as 'same'.
4. *Optional*: include `Dropout` in some layers to avoid overfitting.

<img src="https://drive.google.com/uc?id=1VjFRRspKEyzGaT1GoCCkK3IhyXjmqvup" width="700" align="center">

In [ ]:
def unet(input_size = (128,128,1)):
    inputs = Input(input_size)
    
    # YOUR CODE HERE #
    # Encoder layer 1

    # Encoder layer 2
    
    # Encoder layer 3
    
    # ...   
    # Center (compressed representation)    
    # ...
    # Decoder layer 1

    # Decoder layer 2    

    # Decoder layer 3    

    output = Conv2D(1, 1, activation = 'sigmoid')(conv3)
    model = Model(inputs = inputs, outputs = output)
        
    return model

    

In [ ]:
# replace the unet_simple with the unet() model you create
#model = unet()
model = unet_simple()
model.summary()

Compile the model:

In [ ]:
model.compile(loss = "binary_crossentropy", optimizer = "nadam", metrics = ["accuracy"])

The next cell code is how to fit the model. However, it will take too long, so:

In [ ]:
monitor = "accuracy"
early_stopping = EarlyStopping(monitor = monitor, patience = 100, verbose = 1)
model_checkpoint = ModelCheckpoint("my_model.model", monitor = monitor, save_best_only = True, verbose = 1)
reduce_lr = ReduceLROnPlateau(monitor = monitor, factor = 0.1, patience = 5, min_lr = 0.00000005, verbose = 1)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

epochs = 20
batch_size = 20

history = model.fit(x_train, y_train,
                    validation_data = (x_valid, y_valid), 
                    epochs = epochs,
                    batch_size = batch_size,
                    callbacks = [early_stopping, model_checkpoint, reduce_lr, tensorboard_callback],
                    verbose = 2)

In [ ]:
type(history)

In [ ]:
historypd = pd.DataFrame(history.history)
historypd

Let's check how the model was trained.

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(15,5))
ax_loss.plot(historypd.index, historypd.loss, label="Train loss")
ax_loss.plot(historypd.index, historypd.val_loss, label="Validation loss")
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Loss')
ax_loss.legend()
ax_acc.plot(historypd.index, historypd.accuracy, label="Train accuracy")
ax_acc.plot(historypd.index, historypd.val_accuracy, label="Validation accuracy")
ax_acc.set_xlabel('Epoch')
ax_acc.set_ylabel('Accuracy')
ax_acc.legend()

### Predictions on the validation sets

In [ ]:
preds_valid = model.predict(x_valid)
preds_valid = resizeImage(preds_valid[:,:,:,0], size = 101)
y_valid_ori = resizeImage(y_valid[:,:,:,0], size = 101)
x_valid_ori = resizeImage(x_valid[:,:,:,0], size = 101)

In [ ]:
def plotImageTranspValid(file1, file2, file3, k, alpha1 = 0.2, alpha2 = 0.2):
    fig, ax = plt.subplots(nrows=k, ncols=k, figsize=(18, 18))
    for i in range(k):
        for j in range(k):
            ind = randint(0,file1.shape[0]-1)
            ax[i,j].imshow(file1[ind], cmap='Greys')
            ax[i,j].imshow(file2[ind], cmap='Greens', alpha = alpha1)
            ax[i,j].imshow(file3[ind], cmap='Reds', alpha = alpha2)
            ax[i,j].set_axis_off()
    fig.subplots_adjust(wspace = -0.15, hspace = 0.02)
    plt.suptitle("Green: salt, Red: prediction.")
    return(fig)

In [ ]:
fig = plotImageTranspValid(x_valid_ori, y_valid_ori, np.round(preds_valid), k = 8, alpha1 = 0.3, alpha2 = 0.3)

### Scoring Metric

Score the model predictions and select an optimized threshold. Using the [IoU](https://en.wikipedia.org/wiki/Jaccard_index) (intersection over union) score metric.

<img src="https://drive.google.com/uc?id=1ISt6kdojCNHKJujTfzVJ3ijlytmZRRpR" width="200" width="200" align="center">


In [ ]:
def iou_metric(y_true_in, y_pred_in, print_table=False):
    labels = y_true_in
    y_pred = y_pred_in
    
    true_objects = 2
    pred_objects = 2

    intersection = np.histogram2d(labels.flatten(), y_pred.flatten(), bins=(true_objects, pred_objects))[0]

    # Compute areas (needed for finding the union between all objects)
    area_true = np.histogram(labels, bins = true_objects)[0]
    area_pred = np.histogram(y_pred, bins = pred_objects)[0]
    area_true = np.expand_dims(area_true, -1)
    area_pred = np.expand_dims(area_pred, 0)

    # Compute union
    union = area_true + area_pred - intersection

    # Exclude background from the analysis
    intersection = intersection[1:,1:]
    union = union[1:,1:]
    union[union == 0] = 1e-9

    # Compute the intersection over union
    iou = intersection / union

    # Precision helper function
    def precision_at(threshold, iou):
        matches = iou > threshold
        true_positives = np.sum(matches, axis=1) == 1   # Correct objects
        false_positives = np.sum(matches, axis=0) == 0  # Missed objects
        false_negatives = np.sum(matches, axis=1) == 0  # Extra objects
        tp, fp, fn = np.sum(true_positives), np.sum(false_positives), np.sum(false_negatives)
        return tp, fp, fn

    # Loop over IoU thresholds
    prec = []
    if print_table:
        print("Thresh\tTP\tFP\tFN\tPrec.")
    for t in np.arange(0.5, 1.0, 0.05):
        tp, fp, fn = precision_at(t, iou)
        if (tp + fp + fn) > 0:
            p = tp / (tp + fp + fn)
        else:
            p = 0
        if print_table:
            print("{:1.3f}\t{}\t{}\t{}\t{:1.3f}".format(t, tp, fp, fn, p))
        prec.append(p)
    
    if print_table:
        print("AP\t-\t-\t-\t{:1.3f}".format(np.mean(prec)))
    return np.mean(prec)

def iou_metric_batch(y_true_in, y_pred_in):
    batch_size = y_true_in.shape[0]
    metric = []
    for batch in range(batch_size):
        value = iou_metric(y_true_in[batch], y_pred_in[batch])
        metric.append(value)
    return np.mean(metric)

In [ ]:
thresholds = np.linspace(0, 1, 50)
ious = np.array([iou_metric_batch(y_valid_ori, np.int32(preds_valid > threshold)) for threshold in thresholds])

In [ ]:
threshold_best_index = np.argmax(ious[9:-10]) + 9
iou_best = ious[threshold_best_index]
threshold_best = thresholds[threshold_best_index]

In [ ]:
plt.plot(thresholds, ious)
plt.plot(threshold_best, iou_best, "xr", label="Best threshold")
plt.xlabel("Threshold")
plt.ylabel("IoU")
plt.title("Threshold vs IoU ({}, {})".format(threshold_best, iou_best))
plt.legend()

In [ ]:
fig = plotImageTranspValid(x_valid_ori, y_valid_ori, np.int32(preds_valid > threshold_best), k = 8, alpha1 = 0.3, alpha2 = 0.3)

Below is the prediction we got for the competition and our score was $0.837$.

<img src="https://drive.google.com/uc?id=1R0S9K2j3ePT3hiBONuqBJCJyjGmAwOnF" width="1400" align="center">